# YouTube → Whisper 字幕（Colab）

用 faster-whisper 把沒有字幕的 YouTube 影片轉成 `.srt` / `.txt`。

**使用前：** 選單「執行階段 → 變更執行階段類型 → T4 GPU」，然後依序執行每一格。

每一步失敗都會印出原因與解法；看不懂就把那格的完整輸出貼給 Claude。

In [ ]:
#@title 1. 安裝套件
!pip -q install -U yt-dlp faster-whisper opencc-python-reimplemented
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)

import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode == 0:
    print("✅ GPU:", gpu.stdout.strip())
else:
    print("⚠️ 沒有 GPU，會用 CPU 跑（很慢）。請到「執行階段 → 變更執行階段類型」選 T4 GPU 後重新執行。")

In [ ]:
#@title 2. 參數
URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  #@param {type:"string"}
MODEL = "large-v3"  #@param ["large-v3", "medium", "small", "base"]
LANGUAGE = "auto"  #@param ["auto", "zh", "en", "ja"]
TO_TRADITIONAL = True  #@param {type:"boolean"}

In [ ]:
#@title 3. 下載音訊（被 YouTube 擋的話會改請你上傳 cookies 或音檔）
import os, glob
from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError

os.makedirs("audio", exist_ok=True)
for f in glob.glob("audio/*"):
    os.remove(f)

def download(cookiefile=None):
    opts = {
        "format": "bestaudio/best",
        "outtmpl": "audio/%(id)s.%(ext)s",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "m4a"}],
        "quiet": True, "no_warnings": True,
    }
    if cookiefile:
        opts["cookiefile"] = cookiefile
    with YoutubeDL(opts) as ydl:
        info = ydl.extract_info(URL, download=True)
    return info.get("title", info["id"])

AUDIO, TITLE = None, None
try:
    TITLE = download("cookies.txt" if os.path.exists("cookies.txt") else None)
except DownloadError as e:
    msg = str(e)
    print("❌ 下載失敗：", msg[:500])
    if "not a bot" in msg or "Sign in" in msg or "403" in msg:
        print("""
YouTube 擋了 Colab 的 IP。二選一：
  A) 在自己的 Chrome 裝「Get cookies.txt LOCALLY」擴充功能，登入 YouTube 後匯出 cookies.txt，下面上傳它
  B) 直接上傳影片/音檔（mp3、m4a、mp4 都可以）
""")
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    if name.endswith(".txt"):
        os.replace(name, "cookies.txt")
        TITLE = download("cookies.txt")
    else:
        os.replace(name, os.path.join("audio", name))
        TITLE = os.path.splitext(name)[0]

AUDIO = glob.glob("audio/*")[0]
print("✅ 音檔：", AUDIO, "|", TITLE)

In [ ]:
#@title 4. Whisper 轉錄（GPU 失敗或記憶體不足會自動退回）
import gc
from faster_whisper import WhisperModel

def load(model_name):
    try:
        return WhisperModel(model_name, device="cuda", compute_type="float16"), "cuda"
    except Exception as e:
        print(f"⚠️ GPU 載入失敗（{type(e).__name__}: {str(e)[:200]}），改用 CPU")
        return WhisperModel(model_name, device="cpu", compute_type="int8"), "cpu"

def transcribe(model_name):
    model, device = load(model_name)
    print(f"▶ 模型 {model_name} @ {device}")
    segs, info = model.transcribe(
        AUDIO,
        language=None if LANGUAGE == "auto" else LANGUAGE,
        vad_filter=True,
        beam_size=5,
        initial_prompt="以下是繁體中文的訪談逐字稿。" if LANGUAGE == "zh" else None,
    )
    print(f"偵測語言：{info.language}（{info.language_probability:.0%}），長度 {info.duration/60:.1f} 分鐘")
    out = []
    for s in segs:  # generator：真正的運算在這裡發生
        out.append(s)
        print(f"[{s.start:7.1f}s] {s.text.strip()}")
    return out

fallback = ["large-v3", "medium", "small", "base"]
SEGMENTS = None
for m in fallback[fallback.index(MODEL):]:
    try:
        SEGMENTS = transcribe(m)
        break
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print(f"⚠️ {m} 失敗（{str(e)[:200]}），換小一號模型")
            gc.collect()
            continue
        raise
assert SEGMENTS, "所有模型都失敗了，請把上面的輸出貼給 Claude"

In [ ]:
#@title 5. 輸出 .srt / .txt 並下載
import re
from google.colab import files

conv = None
if TO_TRADITIONAL:
    import opencc
    conv = opencc.OpenCC("s2twp")

def t(text):
    text = text.strip()
    return conv.convert(text) if conv else text

def ts(sec):
    ms = int(round(sec * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

base = re.sub(r'[\\/:*?"<>|]', "_", TITLE)[:80]
with open(f"{base}.srt", "w", encoding="utf-8") as f:
    for i, s in enumerate(SEGMENTS, 1):
        f.write(f"{i}\n{ts(s.start)} --> {ts(s.end)}\n{t(s.text)}\n\n")
with open(f"{base}.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(t(s.text) for s in SEGMENTS))

print("✅ 完成：", f"{base}.srt", f"{base}.txt")
files.download(f"{base}.srt")
files.download(f"{base}.txt")